In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Curved_Template_Reacquisition_v1'
BRANCH='lcx-curved-template-reacquisition-from-main'


# OpenPlaque — LCX curved-template reacquisition — cache-only v3

This version does **not** use `Full_DICOM.zip`. It loads the historical curved RCA/LCX CT inputs directly from the cached NIfTI files in `UCLA_Plaque_Context_Verification`, together with the cached nnU-Net masks. The workflow runs in a fresh subprocess with explicit `PYTHONPATH`, and any child-process failure prints full stdout/stderr.


In [ ]:
import os, shutil, subprocess, sys, textwrap
repo='/content/OpenPlaque'
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo,'pytest','scikit-image'], check=True)
env=os.environ.copy()
env['PYTHONPATH']=repo + '/src'
print('COMMIT:')
subprocess.run(['git','-C',repo,'rev-parse','HEAD'], check=True)
print('IMPORT CHECK:')
subprocess.run([sys.executable,'-c','import openplaque; print(openplaque.__file__)'], check=True, env=env)
print('SYNTAX CHECK:')
subprocess.run([sys.executable,'-m','py_compile',repo + '/src/openplaque/lcx_curved_template_reacquisition.py',repo + '/src/openplaque/lcx_curved_template_reacquisition_v2.py'], check=True, env=env)
print('TESTS:')
subprocess.run([sys.executable,'-m','pytest','-q',repo + '/tests/test_lcx_curved_template_reacquisition.py',repo + '/tests/test_lcx_curved_template_reacquisition_v2.py'], check=True, env=env)
runner = textwrap.dedent(f'''
from openplaque.lcx_curved_template_reacquisition_v2 import synthetic_lcx_template_v2_self_test, run
print('SELF TEST:', synthetic_lcx_template_v2_self_test(), flush=True)
result = run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})
print('STATUS:', result['summary']['status'], flush=True)
print('REPORT:', result['report'], flush=True)
print('ZIP:', result['zip'], flush=True)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip', flush=True)
''')
print('WORKFLOW:')
proc=subprocess.run([sys.executable,'-c',runner], env=env, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print('--- WORKFLOW STDERR ---')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'LCX workflow failed with exit code {proc.returncode}; full traceback is printed above')
